In this notebook I will create a training table equivalent to ml_model_run_details

# Define Library

In [1]:
# %% [markdown]
# # Jupyter Notebook Loading Header
#
# This is a custom loading header for Jupyter Notebooks in Visual Studio Code.
# It includes common imports and settings to get you started quickly.
# %% [markdown]
## Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
from google.cloud import storage
import os
import tempfile
import time
from datetime import datetime
import uuid
import joblib
import uuid

import gcsfs
import duckdb as dd
import pickle
import joblib
from typing import Union
import io
path = r'C:\Users\Dwaipayan\AppData\Roaming\gcloud\application_default_credentials.json'
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = path
client = bigquery.Client(project='prj-prod-dataplatform')
os.environ["GOOGLE_CLOUD_PROJECT"] = "prj-prod-dataplatform"

# %% [markdown]
## Configure Settings
# Set options or configurations as needed
pd.set_option('display.max_columns', None)
pd.set_option("Display.max_rows", 100)

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\auth\_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


### Function

#### expand_calc_features

In [2]:
import pandas as pd
import json

def expand_calc_features(df):
    """
    Expand the calcFeatures JSON column into separate columns and return the complete DataFrame.

    Parameters:
    df (pd.DataFrame): Input DataFrame with calcFeatures column containing JSON data

    Returns:
    pd.DataFrame: Expanded DataFrame with all original columns plus JSON features as separate columns
    """

    # Make a copy to avoid modifying the original DataFrame
    df_expanded = df.copy()

    # Parse the calcFeatures JSON column
    calc_features_list = []

    for idx, calc_features_str in enumerate(df['calcFeatures']):
        try:
            # Parse the JSON string
            features_dict = json.loads(calc_features_str.replace("'", '"'))  # Replace single quotes with double quotes for valid JSON
            calc_features_list.append(features_dict)
        except (json.JSONDecodeError, AttributeError) as e:
            # If parsing fails, create an empty dict and print warning
            print(f"Warning: Could not parse calcFeatures at index {idx}: {e}")
            calc_features_list.append({})

    # Create DataFrame from the parsed JSON data
    calc_features_df = pd.DataFrame(calc_features_list)

    # Add prefix to JSON-derived columns to avoid conflicts
    calc_features_df = calc_features_df.add_prefix('calc_')

    # Reset index to ensure proper alignment
    df_expanded = df_expanded.reset_index(drop=True)
    calc_features_df = calc_features_df.reset_index(drop=True)

    # Combine original DataFrame with expanded calcFeatures
    result_df = pd.concat([df_expanded, calc_features_df], axis=1)

    return result_df


#### expand_calc_features_robust

In [3]:
import pandas as pd
import json

def expand_calc_features_robust(df):
    """
    Expand the calcFeatures JSON column into separate columns with better error handling.

    Parameters:
    df (pd.DataFrame): Input DataFrame with calcFeatures column containing JSON data

    Returns:
    pd.DataFrame: Expanded DataFrame with all original columns plus JSON features as separate columns
    """

    # Make a copy to avoid modifying the original DataFrame
    df_expanded = df.copy()

    # Parse the calcFeatures JSON column
    calc_features_data = []

    for idx, row in df.iterrows():
        calc_features_str = row['calcFeatures']

        if pd.isna(calc_features_str) or calc_features_str == '':
            calc_features_data.append({})
            continue

        try:
            # Clean the string and parse JSON
            cleaned_str = calc_features_str.replace("'", '"').replace('None', 'null').replace('True', 'true').replace('False', 'false')
            features_dict = json.loads(cleaned_str)
            calc_features_data.append(features_dict)
        except Exception as e:
            print(f"Warning: Could not parse calcFeatures at index {idx}: {e}")
            print(f"Problematic string: {calc_features_str[:100]}...")  # Print first 100 chars
            calc_features_data.append({})

    # Create DataFrame from the parsed JSON data
    calc_features_df = pd.DataFrame(calc_features_data)

    # Add prefix to JSON-derived columns to avoid conflicts with existing columns
    calc_features_df = calc_features_df.add_prefix('feat_')

    # Combine DataFrames
    result_df = pd.concat([df_expanded, calc_features_df], axis=1)

    print(f"Original DataFrame shape: {df.shape}")
    print(f"Expanded DataFrame shape: {result_df.shape}")
    print(f"Added {len(calc_features_df.columns)} new columns from calcFeatures")

    return result_df

# Table Name

In [4]:
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"

# Transform data v1.2

In [5]:
import pandas as pd
import json
import uuid
from datetime import datetime
from typing import List

def transform_data_v1_2(
    d1: pd.DataFrame, 
    feature_column: List[str], 
    a: str = 'demo_score', 
    modelDisplayName: str = 'Cash_beta_trench1_Demo_backscore', 
    tc: str = "", 
    subscription_name: str = 'sil_march 25 models'
) -> pd.DataFrame:
    """
    Transforms input data into a structured format suitable for model scoring output.

    Parameters:
    - d1 (pd.DataFrame): Input DataFrame containing raw data.
    - feature_column (List[str]): List of column names to include in the 'calcFeature' JSON.
    - a (str): Column name containing the prediction score. Default is 'demo_score'.
    - modelDisplayName (str): Name of the model used for scoring.
    - tc (str): Trench category (optional).
    - do (str): Device operating system. Default is 'android'.
    - subscription_name (str): Name of the subscription or model group.

    Returns:
    - pd.DataFrame: Transformed DataFrame with structured output.
    """

    # Make a copy of the input DataFrame to avoid modifying the original
    df = d1.copy()
    
    # Initialize an empty list to store transformed rows
    output_data = []
    
    # Iterate over each row in the DataFrame
    for _, row in df.iterrows():
        # Initialize dictionary to hold feature values
        calc_feature = {}
        
        # Loop through each feature column and extract its value from the row
        for col in feature_column:
            if col in row and pd.notna(row[col]):
                # Convert datetime values to ISO format strings
                if isinstance(row[col], pd.Timestamp):
                    calc_feature[col] = row[col].isoformat()
                else:
                    calc_feature[col] = row[col]
        
        # Get the current timestamp for start_time, end_time, and publish_time
        current_time = datetime.now().isoformat()
        
        # Construct the output row dictionary with required fields
        output_row = {
            "customerId": row['customer_id'],  # Unique customer identifier
            "digitalLoanAccountId": row['digitalLoanAccountId'],  # Loan account ID
            "crifApplicationId": str(uuid.uuid4()),  # Random UUID for application ID
            "prediction": row.get(a, 0),  # Prediction score from specified column
            "start_time": current_time,  # Timestamp when processing starts
            "end_time": current_time,    # Timestamp when processing ends
            "modelDisplayName": modelDisplayName,  # Name of the model used
            "modelVersionId": "v1.2",  # Static model version
            "calcFeature": json.dumps(calc_feature, default=str),  # Features as JSON string
            "subscription_name": subscription_name,  # Subscription name
            "message_id": str(uuid.uuid4()),  # Random UUID for message ID
            "publish_time": current_time,  # Timestamp when message is published
            "attributes": "{}",  # Placeholder for additional attributes
            "trenchCategory": tc,  # Optional trench category
            "deviceOs": row['osType'],
            "Data_selection": row['Data_selection'],  # Data selection
            "Application_date": row['application_date'],
        }
        
        # Append the transformed row to the output list
        output_data.append(output_row)
    
    # Convert the list of dictionaries to a DataFrame
    output_df = pd.DataFrame(output_data)
    
    # Return the transformed DataFrame
    return output_df


# Version 1.2

## Cash

## Transaction Score Trench 2 V1.2

### Trench 2

In [6]:
sq = """WITH
  base AS (
    SELECT
      r.customer_id customerId,
      r.digitalLoanAccountId,
      r.ln_loan_appln_time application_submission_date,
      'Trench 2' as trenchCategory,
        r.tx_c_transaction_score_t2,
        r.tx_meng_ql_calculator_tot_visit_cnt,
        r.tx_first_product_user_segment_WOE,
        r.tx_first_applied_loan_type_bin_WOE,
        r.tx_cnt_rejected_loans,	
        r.tx_appsflyer_install_to_registration_minutes,	
        r.first_applied_loan_amount,	
        r.tx_deposit_accnt_cnt,
        r.tx_cnt_cash_in_total,
        r.tx_cnt_incomplete_loan_apps,
        r.tx_amt_cash_in_total,
        r.tx_last_applied_loan_tenor_bin_WOE,
        case when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%andro%' then 'android'
              when lower(coalesce(lmt.osversion_v2, lmt.osVersion)) like '%os%' then 'ios'
              when lower(lmt.deviceType) like '%andro%' then 'android'
              else 'ios' end osType,
      date(
        IF(
          lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) as application_date,
      CASE
        WHEN date(DATETIME(r.ln_loan_appln_time, 'Asia/Manila')) between '2024-09-01' and '2025-02-28' then 'Dev_Train'
        WHEN date(DATETIME(r.ln_loan_appln_time, 'Asia/Manila')) < '2024-09-01' then 'Pre_Train'
        else 'Dev_Test' END Data_selection
    FROM  `worktable_data_analysis.cash_gamma_txn_all_applied_backscored_20260826` r
    LEFT JOIN `risk_credit_mis.loan_master_table` lmt
      ON lmt.digitalLoanAccountId = r.digitalLoanAccountId
  )
SELECT
  customerId customer_id,
  digitalLoanAccountId,
  application_submission_date,
  trenchCategory,
  tx_c_transaction_score_t2,
  tx_meng_ql_calculator_tot_visit_cnt,
  tx_first_product_user_segment_WOE,
  tx_first_applied_loan_type_bin_WOE,
  tx_cnt_rejected_loans,	
  tx_appsflyer_install_to_registration_minutes,	
  first_applied_loan_amount,	
  tx_deposit_accnt_cnt,
  tx_cnt_cash_in_total,
  tx_cnt_incomplete_loan_apps,
  tx_amt_cash_in_total,
  tx_last_applied_loan_tenor_bin_WOE,
  osType,
  application_date,
  Data_selection
FROM base
WHERE
  trenchCategory = 'Trench 2'
  AND tx_c_transaction_score_t2 IS NOT NULL;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")



Job ID 67227c2b-f714-4b9a-add2-fc420053144b successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (387099, 19)


In [7]:
feature_column = ['tx_c_transaction_score_t2',
       'tx_meng_ql_calculator_tot_visit_cnt',
       'tx_first_product_user_segment_WOE',
       'tx_first_applied_loan_type_bin_WOE', 'tx_cnt_rejected_loans',
       'tx_appsflyer_install_to_registration_minutes',
       'first_applied_loan_amount', 'tx_deposit_accnt_cnt',
       'tx_cnt_cash_in_total', 'tx_cnt_incomplete_loan_apps',
       'tx_amt_cash_in_total', 'tx_last_applied_loan_tenor_bin_WOE']

dfd = transform_data_v1_2(data, feature_column, a='tx_c_transaction_score_t2', modelDisplayName='Transaction_Score_Cash_Model', tc='Trench 2', subscription_name = 'Transaction Cash v1.2') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

the shape of the transformed dataframe is:	 (387099, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,1983552,10305fe1-bc54-42df-b934-4e20cb1b560b,21808bac-5e47-445f-9fce-4a06e52e405b,0.601637,2026-09-24T15:23:12.167109,2026-09-24T15:23:12.167109,Transaction_Score_Cash_Model,v1.2,"{""tx_c_transaction_score_t2"": 0.60163722463902...",Transaction Cash v1.2,a75048dc-06b8-40ef-9912-e850bbf4c19f,2026-09-24T15:23:12.167109,{},Trench 2,android,Dev_Test,2026-03-17
1,2665808,f299a84d-d309-4c42-b1ba-607be4f6831a,64e5134a-0c8a-4042-b64f-eb46eee98bde,0.529895,2026-09-24T15:23:12.168240,2026-09-24T15:23:12.168240,Transaction_Score_Cash_Model,v1.2,"{""tx_c_transaction_score_t2"": 0.52989486070670...",Transaction Cash v1.2,0d44f4b0-a76d-4ef0-aaf2-d0f3f7b4342a,2026-09-24T15:23:12.168240,{},Trench 2,android,Dev_Test,2025-12-16
2,2060720,8f7a4532-4cbd-434e-ab2d-fa7c5105cf2e,673f6661-ad18-4fe4-b01e-e1cb2890ab74,0.631991,2026-09-24T15:23:12.168810,2026-09-24T15:23:12.168810,Transaction_Score_Cash_Model,v1.2,"{""tx_c_transaction_score_t2"": 0.63199084051657...",Transaction Cash v1.2,02c8916f-1d41-46d9-94df-0f583e79f080,2026-09-24T15:23:12.168810,{},Trench 2,android,Dev_Test,2025-12-17
3,2064642,e1647d73-938c-4876-bf9b-af4318b63f89,4118fe2a-fe1f-4820-97d2-1b50110a7a96,0.445134,2026-09-24T15:23:12.169368,2026-09-24T15:23:12.169368,Transaction_Score_Cash_Model,v1.2,"{""tx_c_transaction_score_t2"": 0.44513360920475...",Transaction Cash v1.2,7369686c-a659-434d-a39a-5f7b1097261e,2026-09-24T15:23:12.169368,{},Trench 2,android,Dev_Test,2025-12-19
4,1102539,fec0461b-771e-4ae5-844b-af28a4e7a501,b6f4e6bc-a0ef-4c47-9004-0b314b67bd51,0.715232,2026-09-24T15:23:12.169368,2026-09-24T15:23:12.169368,Transaction_Score_Cash_Model,v1.2,"{""tx_c_transaction_score_t2"": 0.71523215309899...",Transaction Cash v1.2,04efba8d-7ae0-484b-a5d7-753d6bd374dd,2026-09-24T15:23:12.169368,{},Trench 2,android,Dev_Test,2025-12-18


In [8]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,331543,2025-03-01,2026-08-25
1,Dev_Train,55556,2024-09-01,2025-02-28


In [9]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=4c094f4c-e307-4061-8485-ff4744145011>

## Transaction Score Trench 3 V1.2

### Trench 3

In [10]:
sq = """
WITH
  base AS (
    SELECT
      r.customer_id customerId,
      r.digitalLoanAccountId,
      r.ln_loan_appln_time application_submission_date,
      'Trench 3' as trenchCategory,
      r.cash_trx_score,
      r.tx_med_days_bt_cash_out_trans,
      r.tx_cnt_installments_paid_tot_with_dpd,
      r.time_since_last_applied_loan_application_time,
      r.cnt_jira_tickets_created_bin,
      r.dob_observation_date,
      r.tx_min_age_completed_loans,
      r.last_applied_loan_decision,
      r.tx_amt_cash_in_total,
      r.last_applied_loan_type_bin,
      r.tx_cnt_completed_loans,
      r.tx_max_ever_dpd,
      r.meng_no_of_logins,
      r.last_applied_loan_tenor,
      r.tx_avg_days_bt_cash_in_trans,
      lower(r.ln_os_type)  osType,
      date(
        IF(
          lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) as application_date,
      CASE
        WHEN date(DATETIME(r.ln_loan_appln_time, 'Asia/Manila')) between '2024-01-01' and '2024-11-30' then 'Dev_Train'
        WHEN date(DATETIME(r.ln_loan_appln_time, 'Asia/Manila')) < '2024-01-01' then 'Pre_Train'
        else 'Dev_Test' END Data_selection
    FROM  `worktable_data_analysis.cash_trench3_transaction_all_applied_backscored_20240101_20260825` r
    LEFT JOIN `risk_credit_mis.loan_master_table` lmt
      ON lmt.digitalLoanAccountId = r.digitalLoanAccountId
  )
SELECT
  safe_cast(customerId as int64)  customer_id,
  digitalLoanAccountId,
  application_submission_date,
  trenchCategory,
  cash_trx_score,
  tx_med_days_bt_cash_out_trans,
  tx_cnt_installments_paid_tot_with_dpd,
  time_since_last_applied_loan_application_time,
  cnt_jira_tickets_created_bin,
  dob_observation_date,
  tx_min_age_completed_loans,
  last_applied_loan_decision,
  tx_amt_cash_in_total,
  last_applied_loan_type_bin,
  tx_cnt_completed_loans,
  tx_max_ever_dpd,
  meng_no_of_logins,
  last_applied_loan_tenor,
  tx_avg_days_bt_cash_in_trans,
  osType,
  application_date,
  Data_selection
FROM base
WHERE
  trenchCategory = 'Trench 3'
  AND cash_trx_score IS NOT NULL;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")



Job ID 47f0330f-a789-4e08-9aaa-197d5eb8eb4d successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (199671, 22)


In [11]:
data.columns

Index(['customer_id', 'digitalLoanAccountId', 'application_submission_date',
       'trenchCategory', 'cash_trx_score', 'tx_med_days_bt_cash_out_trans',
       'tx_cnt_installments_paid_tot_with_dpd',
       'time_since_last_applied_loan_application_time',
       'cnt_jira_tickets_created_bin', 'dob_observation_date',
       'tx_min_age_completed_loans', 'last_applied_loan_decision',
       'tx_amt_cash_in_total', 'last_applied_loan_type_bin',
       'tx_cnt_completed_loans', 'tx_max_ever_dpd', 'meng_no_of_logins',
       'last_applied_loan_tenor', 'tx_avg_days_bt_cash_in_trans', 'osType',
       'application_date', 'Data_selection'],
      dtype='object')

In [12]:
feature_column = ['cash_trx_score', 'tx_med_days_bt_cash_out_trans',
       'tx_cnt_installments_paid_tot_with_dpd',
       'time_since_last_applied_loan_application_time',
       'cnt_jira_tickets_created_bin', 'dob_observation_date',
       'tx_min_age_completed_loans', 'last_applied_loan_decision',
       'tx_amt_cash_in_total', 'last_applied_loan_type_bin',
       'tx_cnt_completed_loans', 'tx_max_ever_dpd', 'meng_no_of_logins',
       'last_applied_loan_tenor', 'tx_avg_days_bt_cash_in_trans']

dfd = transform_data_v1_2(data, feature_column, a='cash_trx_score', modelDisplayName='Transaction_Score_Cash_Model', tc='Trench 3', subscription_name = 'Transaction Cash v1.2') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

the shape of the transformed dataframe is:	 (199671, 17)


,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,1910984,7d67faad-ddea-41c6-adfa-80981c12eac1,4896a253-083b-458a-bab6-010c346a77eb,0.542823,2026-09-24T15:26:32.210556,2026-09-24T15:26:32.210556,Transaction_Score_Cash_Model,v1.2,"{""cash_trx_score"": 0.5428228861239821, ""tx_med...",Transaction Cash v1.2,e2320570-917f-4eee-b556-920e331d6569,2026-09-24T15:26:32.210556,{},Trench 3,ios,Dev_Train,2024-10-02
1,1610823,fe8720cb-9c19-46f5-bca4-5b6b73b49e09,03c0d10f-c46e-4816-a260-d45e693b9e72,0.415462,2026-09-24T15:26:32.212063,2026-09-24T15:26:32.212063,Transaction_Score_Cash_Model,v1.2,"{""cash_trx_score"": 0.4154619465492977, ""tx_med...",Transaction Cash v1.2,2211e448-23fa-43f9-901e-3e6dfc571fe8,2026-09-24T15:26:32.212063,{},Trench 3,android,Dev_Train,2024-01-19
2,1325811,29fcf66e-591e-4092-a688-edb1502378d5,3b63e288-4283-4867-97fd-90ab38d1c52e,0.465137,2026-09-24T15:26:32.213069,2026-09-24T15:26:32.213069,Transaction_Score_Cash_Model,v1.2,"{""cash_trx_score"": 0.4651370971727082, ""tx_med...",Transaction Cash v1.2,40d42696-a631-4d46-a567-9d97c15ad2e5,2026-09-24T15:26:32.213069,{},Trench 3,android,Dev_Train,2024-02-28
3,1047855,21a42889-82c0-4c37-bfc8-47b9eda42298,e1eba5b8-4855-4999-9cd8-68c0a46d7344,0.594090,2026-09-24T15:26:32.213069,2026-09-24T15:26:32.213069,Transaction_Score_Cash_Model,v1.2,"{""cash_trx_score"": 0.5940903073517706, ""tx_med...",Transaction Cash v1.2,d2ad6165-2ef7-4e44-bd64-3fd798fcd4b5,2026-09-24T15:26:32.213069,{},Trench 3,ios,Dev_Train,2024-03-25
4,1633693,48c61083-c944-4322-9527-03569ef9190f,254a7dd3-0c19-463f-85bc-bbc4efb0ed34,0.554805,2026-09-24T15:26:32.214070,2026-09-24T15:26:32.214070,Transaction_Score_Cash_Model,v1.2,"{""cash_trx_score"": 0.5548048614455203, ""tx_med...",Transaction Cash v1.2,65d51211-5c49-448d-b6e7-8ce1b28e1929,2026-09-24T15:26:32.214070,{},Trench 3,ios,Dev_Train,2024-04-01


In [13]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,171896,2024-12-01,2026-09-07
1,Dev_Train,27775,2024-01-01,2024-11-30


In [14]:
dfd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199671 entries, 0 to 199670
Data columns (total 17 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   customerId            199671 non-null  int64  
 1   digitalLoanAccountId  199671 non-null  object 
 2   crifApplicationId     199671 non-null  object 
 3   prediction            199671 non-null  float64
 4   start_time            199671 non-null  object 
 5   end_time              199671 non-null  object 
 6   modelDisplayName      199671 non-null  object 
 7   modelVersionId        199671 non-null  object 
 8   calcFeature           199671 non-null  object 
 9   subscription_name     199671 non-null  object 
 10  message_id            199671 non-null  object 
 11  publish_time          199671 non-null  object 
 12  attributes            199671 non-null  object 
 13  trenchCategory        199671 non-null  object 
 14  deviceOs              199671 non-null  object 
 15  

In [15]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=33662e11-e512-4213-8451-c2cdf221f678>

## alpha_stack_model_cash

### Trench 1

In [21]:
sq = """
DELETE FROM `prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116`
WHERE modelDisplayName = 'Alpha-Cash-Stack-Model'
  AND modelVersionId = 'v1.2'
  AND trenchCategory = 'Trench 1';
"""

query_job = client.query(sq)
query_job.result()

print(f"Deleted {query_job.num_dml_affected_rows} rows.")

Deleted 0 rows.


In [50]:
sq = """
select 
  r.customer_id,
  r.digitalLoanAccountId, 
  r.`cb_demo_score` demo_score, 
  r.`cb_event_score` event_score, 
  r.`c_apps_score` apps_score, 
  r.`ca_cic_score` cic_score, 
  r.`c_credo_score` credo_score, 
  r.`c_device_score` device_score, 
  r.`ca_t1_stack_score` stack_score,
  r.ln_os_type osType,
   date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
  case when lower(r.ln_os_type) like '%android%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2025-12-01' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2026-04-30' then 'Dev_Train'
           when lower(r.ln_os_type) like '%android%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2026-05-01' then 'Dev_Test'
           when lower(r.ln_os_type) like '%ios%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2026-01-01' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) <= '2026-04-30' then 'Dev_Train'
           when lower(r.ln_os_type) like '%ios%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2026-05-01' then 'Dev_Test' 
           end Data_selection
from `worktable_data_analysis.cash_alpha_trench1_applied_loans_backscored_stack_v1_2_202601_202608` r
left join `risk_credit_mis.loan_master_table` loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where r.ca_t1_stack_score is not null
;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 1cb73033-29e4-4076-be2d-d5c0c8c4c115 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (52379, 12)


In [51]:
feature_column = ['demo_score','event_score', 'apps_score', 'credo_score', 'cic_score', 'device_score', 'stack_score']

dfd = transform_data_v1_2(data, feature_column, a='stack_score', modelDisplayName='Alpha-Cash-Stack-Model', tc='Trench 1', subscription_name = 'Cash Jan26toApril26 Model') 
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,4468359,8979feed-481a-4417-8af1-9a1779e207b0,3861c453-a47f-4698-8c94-c88c01e701f4,0.461456,2026-09-25T16:02:33.458078,2026-09-25T16:02:33.458078,Alpha-Cash-Stack-Model,v1.2,"{""demo_score"": 0.42612741247916813, ""event_sco...",Cash Jan26toApril26 Model,2010e28d-09de-46e3-81f4-c5af52c09ebe,2026-09-25T16:02:33.458078,{},Trench 1,Android,Dev_Test,2026-07-12
1,4390369,187b5a5d-28b1-4ba8-af6a-3d854b654c7d,6b7bd14e-adff-496f-8409-df79c2983dd6,0.532959,2026-09-25T16:02:33.458682,2026-09-25T16:02:33.458682,Alpha-Cash-Stack-Model,v1.2,"{""demo_score"": 0.3862764698920403, ""event_scor...",Cash Jan26toApril26 Model,68c94807-153f-4982-95dd-ca7989716c2f,2026-09-25T16:02:33.458682,{},Trench 1,iOS,Dev_Test,2026-06-05
2,4417127,da899520-8226-43fb-ae33-882211278172,3c931905-17ef-4335-8028-4971ccb41b89,0.457134,2026-09-25T16:02:33.459293,2026-09-25T16:02:33.459293,Alpha-Cash-Stack-Model,v1.2,"{""demo_score"": 0.4776823287502559, ""event_scor...",Cash Jan26toApril26 Model,99b9c42e-e906-4ae2-b8d2-69d4b8d597b4,2026-09-25T16:02:33.459293,{},Trench 1,iOS,Dev_Test,2026-06-18
3,4519351,39993fab-79ed-4539-8bdd-f715e989e05f,90d0edb3-a417-4f71-a813-55d13ac437f1,0.275917,2026-09-25T16:02:33.459293,2026-09-25T16:02:33.459293,Alpha-Cash-Stack-Model,v1.2,"{""demo_score"": 0.4609466092671523, ""event_scor...",Cash Jan26toApril26 Model,338c0c94-d88d-407b-8f51-d02711680ff1,2026-09-25T16:02:33.459293,{},Trench 1,iOS,Dev_Test,2026-08-04
4,4475696,e38b8e62-267f-4160-a563-07d40bc5aae4,c5a218b0-d8ed-4fd4-b2b9-c97197da898a,0.462930,2026-09-25T16:02:33.459938,2026-09-25T16:02:33.459938,Alpha-Cash-Stack-Model,v1.2,"{""demo_score"": 0.45972642912765455, ""event_sco...",Cash Jan26toApril26 Model,31ed914c-c818-4b55-af9d-da0ec4b090f9,2026-09-25T16:02:33.459938,{},Trench 1,iOS,Dev_Test,2026-07-15


In [52]:
result = dfd.groupby(['Data_selection', 'deviceOs']).agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,deviceOs,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,Android,2949,2026-05-01,2026-08-24
1,Dev_Test,iOS,3318,2026-05-01,2026-08-25
2,Dev_Train,Android,33477,2026-01-01,2026-04-29
3,Dev_Train,iOS,12602,2026-01-01,2026-04-30


`The train period for android for this model is mentioned to start from December 2025 but the data starts from January 2026 only`

In [53]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=24105bda-921e-43c6-b5f4-3ec124ac7552>

### Trench 2

In [54]:
sq = """
DELETE FROM `prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116`
WHERE modelDisplayName = 'Alpha-Cash-Stack-Model'
  AND modelVersionId = 'v1.2'
  AND trenchCategory = 'Trench 2';
"""

query_job = client.query(sq)
query_job.result()

print(f"Deleted {query_job.num_dml_affected_rows} rows.")

Deleted 30300 rows.


In [55]:
sq = """
select 
  r.customer_id,
  r.digitalLoanAccountId, 
  r.`cb_demo_score` demo_score, 
  r.`cb_event_score` event_score, 
  r.`c_apps_score` apps_score, 
  r.`ca_cic_score` cic_score, 
  r.`c_credo_score` credo_score, 
  r.`c_trx_score` trx_score, 
  r.`ca_t2_stack_score` stack_score,
  r.ln_os_type osType,
   date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
  case when lower(r.ln_os_type) like '%android%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2025-12-01' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2026-04-30' then 'Dev_Train'
           when lower(r.ln_os_type) like '%android%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2026-05-01' then 'Dev_Test'
           when lower(r.ln_os_type) like '%ios%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2026-01-01' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) <= '2026-04-30' then 'Dev_Train'
           when lower(r.ln_os_type) like '%ios%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2026-05-01' then 'Dev_Test' 
           end Data_selection
from `prj-prod-dataplatform.worktable_data_analysis.cash_alpha_trench2_applied_loans_backscored_stack_v1_2_202601_202608` r
left join `risk_credit_mis.loan_master_table` loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where r.ca_t2_stack_score is not null
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


Job ID 620d2439-7b32-4bec-9c1e-4492d64e11a6 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (30300, 12)


In [ ]:
feature_column = ['demo_score','event_score', 'apps_score', 'credo_score', 'cic_score', 'trx_score', 'stack_score']

dfd = transform_data_v1_2(data, feature_column, a='stack_score', modelDisplayName='Alpha-Cash-Stack-Model', tc='Trench 2', subscription_name = 'Cash Jan26toApril26 Model') 
dfd.head()

In [ ]:
result = dfd.groupby(['Data_selection', 'deviceOs']).agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,deviceOs,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,Android,2792,2026-05-01,2026-08-24
1,Dev_Test,iOS,1935,2026-05-01,2026-08-25
2,Dev_Train,Android,16534,2026-01-01,2026-04-29
3,Dev_Train,iOS,9020,2026-01-01,2026-04-30


In [ ]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=69fbc8de-75ec-47d1-a68f-58196ccfdc9c>

### Trench 3

In [ ]:
sq = """
DELETE FROM `prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116`
WHERE modelDisplayName = 'Alpha-Cash-Stack-Model'
  AND modelVersionId = 'v1.2'
  AND trenchCategory = 'Trench 3';
"""

query_job = client.query(sq)
query_job.result()

print(f"Deleted {query_job.num_dml_affected_rows} rows.")

In [ ]:
sq = """select 
  r.customer_id,
  r.digitalLoanAccountId, 
  r.`cb_demo_score` demo_score, 
  r.`cb_event_score` event_score, 
  r.`c_apps_score` apps_score, 
  r.`ca_cic_score` cic_score, 
  r.`c_credo_score` credo_score, 
  r.`c_device_score` device_score, 
  r.`ca_t3_stack_score` stack_score,
  r.ln_os_type osType,
   date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
  case when lower(r.ln_os_type) like '%android%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2025-12-01' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2026-04-30' then 'Dev_Train'
           when lower(r.ln_os_type) like '%android%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2026-05-01' then 'Dev_Test'
           when lower(r.ln_os_type) like '%ios%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2026-01-01' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) <= '2026-04-30' then 'Dev_Train'
           when lower(r.ln_os_type) like '%ios%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2026-05-01' then 'Dev_Test' 
           end Data_selection
from `prj-prod-dataplatform.worktable_data_analysis.cash_alpha_trench3_applied_loans_backscored_stack_v1_2_202601_202608` r
left join `risk_credit_mis.loan_master_table` loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where r.ca_t3_stack_score is not null
;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


Job ID 67a78ddf-bd03-49c3-8cf5-77a1be80d6ee successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (14351, 12)


In [ ]:
feature_column = ['demo_score','event_score', 'apps_score', 'credo_score', 'cic_score', 'device_score', 'stack_score']

dfd = transform_data_v1_2(data, feature_column, a='stack_score', modelDisplayName='Alpha-Cash-Stack-Model', tc='Trench 3', subscription_name = 'Cash Jan26toApril26 Model') 
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3977259,5def0f19-c8f7-45c1-a24c-de496c0f4817,a63e9f53-0730-4836-a83b-5759d29c4832,0.544781,2026-09-25T13:20:35.973340,2026-09-25T13:20:35.973340,Alpha-Cash-Stack-Model,v1.2,"{""demo_score"": 0.4695226436273174, ""credo_scor...",Cash Jan26toApril26 Model,fe4ce4ec-9a02-4762-8e3a-ed9d5a42e5d3,2026-09-25T13:20:35.973340,{},Trench 3,iOS,Dev_Test,2026-05-15
1,3385777,06e10a64-e1e1-46f2-a52b-2cb2361fcb37,a854fc9c-526c-4dac-914a-8926aac5bacb,0.484420,2026-09-25T13:20:35.973340,2026-09-25T13:20:35.973340,Alpha-Cash-Stack-Model,v1.2,"{""demo_score"": 0.47887000166244387, ""credo_sco...",Cash Jan26toApril26 Model,c73bae85-8f49-44e3-90d2-9ca6278d8901,2026-09-25T13:20:35.973340,{},Trench 3,iOS,Dev_Test,2026-05-21
2,3792438,5c35b267-48ae-4bb1-8b31-4a8b6c795d66,5d4b447e-047d-4e2c-992b-0d61b5b4e5f3,0.466974,2026-09-25T13:20:35.973340,2026-09-25T13:20:35.973340,Alpha-Cash-Stack-Model,v1.2,"{""demo_score"": 0.463036743185894, ""credo_score...",Cash Jan26toApril26 Model,1df02ff4-5b7f-40a9-a66e-6ddf2b791340,2026-09-25T13:20:35.973340,{},Trench 3,iOS,Dev_Test,2026-07-06
3,3783458,676b129e-8a8a-4929-b817-add3b1812cb0,29dcda52-32b5-48b5-9929-7ce40325a613,0.455939,2026-09-25T13:20:35.973340,2026-09-25T13:20:35.973340,Alpha-Cash-Stack-Model,v1.2,"{""demo_score"": 0.4189399734413332, ""credo_scor...",Cash Jan26toApril26 Model,9b52d260-d663-48ec-a3a4-c34bab199ed9,2026-09-25T13:20:35.973340,{},Trench 3,iOS,Dev_Train,2026-01-24
4,3798005,89688fac-6ac4-4489-adb0-b00a3d5cdf48,1f942271-6bb3-451b-9d38-fd04fb35a8f8,0.487968,2026-09-25T13:20:35.973340,2026-09-25T13:20:35.973340,Alpha-Cash-Stack-Model,v1.2,"{""demo_score"": 0.43146463878411045, ""credo_sco...",Cash Jan26toApril26 Model,cdc77a98-b043-49a7-a5ed-5ea20614e30c,2026-09-25T13:20:35.973340,{},Trench 3,iOS,Dev_Train,2026-02-26


In [ ]:
result = dfd.groupby(['Data_selection', 'deviceOs']).agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,deviceOs,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,Android,4640,2026-05-01,2026-08-24
1,Dev_Test,iOS,2504,2026-05-01,2026-08-25
2,Dev_Train,Android,4432,2026-01-01,2026-04-29
3,Dev_Train,iOS,2734,2026-01-01,2026-04-30


In [ ]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=31ef53cd-d364-4867-b75c-c5ee79757377>

## Beta-Cash-Stack-Model

### Trench 1

In [ ]:
sq = """
DELETE FROM `prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116`
WHERE modelDisplayName = 'Beta-Cash-Stack-Model'
  AND modelVersionId = 'v1.2'
  AND trenchCategory = 'Trench 1';
"""

query_job = client.query(sq)
query_job.result()

print(f"Deleted {query_job.num_dml_affected_rows} rows.")

In [35]:
sq = """select 
  r.customer_id,
  r.digitalLoanAccountId, 
  r.`cb_demo_score` demo_score, 
  r.`cb_event_score` event_score, 
  r.`c_apps_score` apps_score, 
  r.`c_credo_score` credo_score, 
  r.`cb_t1_stack_score` stack_score,
  r.ln_os_type osType,
   date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
  case when lower(r.ln_os_type) like '%android%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2026-01-01' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2026-04-30' then 'Dev_Train'
           when lower(r.ln_os_type) like '%android%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2026-05-01' then 'Dev_Test'
           when lower(r.ln_os_type) like '%ios%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2026-01-01' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) <= '2026-04-30' then 'Dev_Train'
           when lower(r.ln_os_type) like '%ios%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2026-05-01' then 'Dev_Test' 
           end Data_selection
from prj-prod-dataplatform.worktable_data_analysis.cash_beta_trench1_applied_loans_backscored_stack_v1_2_202601_202608 r
left join `risk_credit_mis.loan_master_table` loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where r.cb_t1_stack_score is not null
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


Job ID 749b38ea-5435-436e-bb87-600cc2839219 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (295904, 10)


In [36]:
feature_column = ['demo_score','event_score', 'apps_score', 'credo_score', 'stack_score']
dfd = transform_data_v1_2(data, feature_column, a='stack_score', modelDisplayName='Beta-Cash-Stack-Model', tc='Trench 1', subscription_name = 'Cash jan26toApril26 Model') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.info()

the shape of the transformed dataframe is:	 (295904, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295904 entries, 0 to 295903
Data columns (total 17 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   customerId            295904 non-null  int64  
 1   digitalLoanAccountId  295904 non-null  object 
 2   crifApplicationId     295904 non-null  object 
 3   prediction            295904 non-null  float64
 4   start_time            295904 non-null  object 
 5   end_time              295904 non-null  object 
 6   modelDisplayName      295904 non-null  object 
 7   modelVersionId        295904 non-null  object 
 8   calcFeature           295904 non-null  object 
 9   subscription_name     295904 non-null  object 
 10  message_id            295904 non-null  object 
 11  publish_time          295904 non-null  object 
 12  attributes            295904 non-null  object 
 13  trenchCategory        295904 non-null  object 


In [37]:
result = dfd.groupby(['Data_selection', 'deviceOs']).agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,deviceOs,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,Android,78170,2026-05-01,2026-08-24
1,Dev_Test,iOS,35768,2026-05-01,2026-08-25
2,Dev_Train,Android,146735,2026-01-01,2026-04-29
3,Dev_Train,iOS,34553,2026-01-01,2026-04-30


In [38]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=76a2c4bb-3ca0-4f59-9b52-db703478401c>

### Trench 2

In [ ]:
sq = """
DELETE FROM `prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116`
WHERE modelDisplayName = 'Beta-Cash-Stack-Model'
  AND modelVersionId = 'v1.2'
  AND trenchCategory = 'Trench 2';
"""

query_job = client.query(sq)
query_job.result()

print(f"Deleted {query_job.num_dml_affected_rows} rows.")

In [42]:
sq = """select 
  r.customer_id,
  r.digitalLoanAccountId, 
  r.`cb_demo_score` demo_score, 
  r.`cb_event_score` event_score, 
  r.`c_apps_score` apps_score, 
  r.c_trx_score trx_score,
  r.`c_credo_score` credo_score, 
  r.c_device_score device_score,
  r.`cb_t2_stack_score` stack_score,
  r.ln_os_type osType,
   date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
  case when lower(r.ln_os_type) like '%android%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2025-12-01' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2026-04-30' then 'Dev_Train'
           when lower(r.ln_os_type) like '%android%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2026-05-01' then 'Dev_Test'
           when lower(r.ln_os_type) like '%ios%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2025-12-01' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) <= '2026-04-30' then 'Dev_Train'
           when lower(r.ln_os_type) like '%ios%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2026-05-01' then 'Dev_Test' 
           end Data_selection
from prj-prod-dataplatform.worktable_data_analysis.cash_beta_trench2_applied_loans_backscored_stack_v1_2_202601_202608 r
left join `risk_credit_mis.loan_master_table` loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where r.cb_t2_stack_score is not null
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


Job ID 14cc4cc0-d33f-4676-9b84-d145de9fd205 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (159673, 12)


In [43]:
feature_column = ['demo_score','event_score', 'apps_score', 'trx_score', 'credo_score', 'device_score', 'stack_score']
dfd = transform_data_v1_2(data, feature_column, a='stack_score', modelDisplayName='Beta-Cash-Stack-Model', tc='Trench 2', subscription_name = 'Cash jan26toApril26 Model') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.info()

the shape of the transformed dataframe is:	 (159673, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159673 entries, 0 to 159672
Data columns (total 17 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   customerId            159673 non-null  int64  
 1   digitalLoanAccountId  159673 non-null  object 
 2   crifApplicationId     159673 non-null  object 
 3   prediction            159673 non-null  float64
 4   start_time            159673 non-null  object 
 5   end_time              159673 non-null  object 
 6   modelDisplayName      159673 non-null  object 
 7   modelVersionId        159673 non-null  object 
 8   calcFeature           159673 non-null  object 
 9   subscription_name     159673 non-null  object 
 10  message_id            159673 non-null  object 
 11  publish_time          159673 non-null  object 
 12  attributes            159673 non-null  object 
 13  trenchCategory        159673 non-null  object 


In [44]:
result = dfd.groupby(['Data_selection', 'deviceOs']).agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,deviceOs,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,Android,54034,2026-05-01,2026-08-24
1,Dev_Test,iOS,24844,2026-05-01,2026-08-25
2,Dev_Train,Android,55958,2026-01-01,2026-04-29
3,Dev_Train,iOS,24366,2026-01-01,2026-04-30


In [45]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=0ad67e94-5a6a-41b6-b25d-6d37cc0a6906>

### Trench 3

In [ ]:
sq = """
DELETE FROM `prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116`
WHERE modelDisplayName = 'Beta-Cash-Stack-Model'
  AND modelVersionId = 'v1.2'
  AND trenchCategory = 'Trench 3';
"""

query_job = client.query(sq)
query_job.result()

print(f"Deleted {query_job.num_dml_affected_rows} rows.")

In [46]:
sq = """select 
  r.customer_id,
  r.digitalLoanAccountId, 
  r.`cb_demo_score` demo_score, 
  r.`cb_event_score` event_score, 
  r.`c_apps_score` apps_score, 
  r.c_trx_score trx_score,
  r.`c_credo_score` credo_score, 
  r.c_device_score device_score,
  r.`cb_t3_stack_score` stack_score,
  r.ln_os_type osType,
   date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
  case when lower(r.ln_os_type) like '%android%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2026-01-01' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2026-04-30' then 'Dev_Train'
           when lower(r.ln_os_type) like '%android%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2026-05-01' then 'Dev_Test'
           when lower(r.ln_os_type) like '%ios%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2026-01-01' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) <= '2026-04-30' then 'Dev_Train'
           when lower(r.ln_os_type) like '%ios%' and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) >= '2026-05-01' then 'Dev_Test' 
           end Data_selection
from prj-prod-dataplatform.worktable_data_analysis.cash_beta_trench3_applied_loans_backscored_stack_v1_2_202601_202608 r
left join `risk_credit_mis.loan_master_table` loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where r.cb_t3_stack_score is not null
;
""" 
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 6236d495-424d-4324-a9d0-85a238edb7c7 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (26915, 12)


In [47]:
feature_column = ['demo_score','event_score', 'apps_score', 'trx_score', 'credo_score', 'device_score', 'stack_score']
dfd = transform_data_v1_2(data, feature_column, a='stack_score', modelDisplayName='Beta-Cash-Stack-Model', tc='Trench 3', subscription_name = 'Cash jan26toApril26 Model') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.info()

the shape of the transformed dataframe is:	 (26915, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26915 entries, 0 to 26914
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   customerId            26915 non-null  int64  
 1   digitalLoanAccountId  26915 non-null  object 
 2   crifApplicationId     26915 non-null  object 
 3   prediction            26915 non-null  float64
 4   start_time            26915 non-null  object 
 5   end_time              26915 non-null  object 
 6   modelDisplayName      26915 non-null  object 
 7   modelVersionId        26915 non-null  object 
 8   calcFeature           26915 non-null  object 
 9   subscription_name     26915 non-null  object 
 10  message_id            26915 non-null  object 
 11  publish_time          26915 non-null  object 
 12  attributes            26915 non-null  object 
 13  trenchCategory        26915 non-null  object 
 14  deviceOs      

In [48]:
result = dfd.groupby(['Data_selection', 'deviceOs']).agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,deviceOs,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,Android,9519,2026-05-01,2026-08-24
1,Dev_Test,iOS,4609,2026-05-01,2026-08-25
2,Dev_Train,Android,8192,2026-01-01,2026-04-29
3,Dev_Train,iOS,4509,2026-01-01,2026-04-30


In [49]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=e83046f6-874b-41f8-bae1-c0e45c425367>

# 🪦💀 Graveyard

#### PSI Functions new

In [ ]:
# ## Updated on 27-10-2025 - Modified for Training Period Baseline
# import pandas as pd
# import numpy as np
# from typing import List, Dict, Tuple
# import warnings
# warnings.filterwarnings('ignore')

# def identify_feature_types(df: pd.DataFrame, feature_list: List[str]) -> Dict[str, List[str]]:
#     """
#     Identify categorical and numerical features from the feature list.

#     Parameters:
#     -----------
#     df : pd.DataFrame
#         Input dataframe
#     feature_list : List[str]
#         List of features to classify

#     Returns:
#     --------
#     Dict with 'categorical' and 'numerical' keys containing respective feature lists
#     """
#     categorical_features = []
#     numerical_features = []

#     for feature in feature_list:
#         if feature not in df.columns:
#             print(f"Warning: Feature '{feature}' not found in dataframe")
#             continue

#         # Check if feature is numeric
#         if pd.api.types.is_numeric_dtype(df[feature]):
#             # If unique values are less than 15 and all integers, treat as categorical
#             unique_vals = df[feature].nunique()
#             if unique_vals < 15 and df[feature].dropna().apply(lambda x: x == int(x) if isinstance(x, (int, float)) else False).all():
#                 categorical_features.append(feature)
#             else:
#                 numerical_features.append(feature)
#         else:
#             categorical_features.append(feature)

#     return {
#         'categorical': categorical_features,
#         'numerical': numerical_features
#     }


# def create_bins_for_features(df: pd.DataFrame,
#                              numerical_features: List[str],
#                              categorical_features: List[str],
#                              train_period_df: pd.DataFrame) -> Dict:
#     """
#     Create bins for numerical features (deciles with fallback) and categorical features (top 6 + others)
#     based on the entire training period data.

#     Parameters:
#     -----------
#     df : pd.DataFrame
#         Full input dataframe
#     numerical_features : List[str]
#         List of numerical features
#     categorical_features : List[str]
#         List of categorical features
#     train_period_df : pd.DataFrame
#         Training period dataframe (June 2024 to March 2025)

#     Returns:
#     --------
#     Dictionary containing binning information for each feature
#     """
#     binning_info = {}

#     # Create bins for numerical features with fallback strategy
#     for feature in numerical_features:
#         valid_data = train_period_df[feature].dropna()

#         if len(valid_data) == 0:
#             binning_info[feature] = {'type': 'numerical', 'bins': None, 'bin_ranges': {}}
#             continue

#         bins = None
#         bin_count = None

#         # Try 10 bins (deciles)
#         try:
#             test_bins = np.percentile(valid_data, np.arange(0, 101, 10))
#             test_bins = np.unique(test_bins)
#             if len(test_bins) >= 11:  # 11 edges = 10 bins
#                 bins = test_bins
#                 bin_count = 10
#         except Exception as e:
#             pass

#         # If 10 bins not possible, try 5 bins
#         if bins is None:
#             try:
#                 test_bins = np.percentile(valid_data, np.arange(0, 101, 20))
#                 test_bins = np.unique(test_bins)
#                 if len(test_bins) >= 6:  # 6 edges = 5 bins
#                     bins = test_bins
#                     bin_count = 5
#             except Exception as e:
#                 pass

#         # If 5 bins not possible, try 3 bins
#         if bins is None:
#             try:
#                 test_bins = np.percentile(valid_data, [0, 33.33, 66.67, 100])
#                 test_bins = np.unique(test_bins)
#                 if len(test_bins) >= 4:  # 4 edges = 3 bins
#                     bins = test_bins
#                     bin_count = 3
#             except Exception as e:
#                 pass

#         # If still no bins possible, use equal distance bins of 5
#         if bins is None:
#             print(f"Warning: Feature '{feature}' has insufficient variance - cannot create standard bins")
#             print(f"Feature '{feature}': Using equal distance bins of 5")

#             min_val = valid_data.min()
#             max_val = valid_data.max()

#             # Create 5 equal distance bins
#             bins = np.linspace(min_val, max_val, 6)  # 6 edges = 5 bins
#             bins = np.unique(bins)
#             bin_count = len(bins) - 1

#             # If all values are the same, add slight buffer
#             if bin_count == 1:
#                 bins = np.array([min_val - 0.1, min_val, min_val + 0.1])
#                 bin_count = 2
#                 print(f"Feature '{feature}': Constant value ({min_val}). Created 2 equal distance bins with buffer")

#         # Add infinity edges to capture all values
#         bins = bins.copy()
#         bins[0] = -np.inf
#         bins[-1] = np.inf

#         print(f"Feature '{feature}': Created {bin_count} bins")

#         # Create bin ranges dictionary
#         bin_ranges = {}
#         for i in range(len(bins)-1):
#             bin_name = f"Bin_{i+1}"
#             bin_ranges[bin_name] = {
#                 'min': bins[i],
#                 'max': bins[i+1],
#                 'range_str': f"[{bins[i]:.2f}, {bins[i+1]:.2f}]" if not np.isinf(bins[i]) and not np.isinf(bins[i+1]) else f"({bins[i]}, {bins[i+1]})"
#             }

#         binning_info[feature] = {
#             'type': 'numerical',
#             'bins': bins,
#             'bin_ranges': bin_ranges,
#             'bin_count': bin_count
#         }

#     # Create bins for categorical features (top 6 + others) using training period
#     for feature in categorical_features:
#         value_counts = train_period_df[feature].value_counts()
#         unique_categories = value_counts.index.tolist()
#         print(f"Unique categories: {unique_categories}")

#         if len(unique_categories) <= 6:
#             # Treat each category as a separate bin
#             top_categories = unique_categories
#         else:
#             # Use top 6 categories only
#             top_categories = value_counts.nlargest(6).index.tolist()

#         print(f"Top categories for feature '{feature}': {top_categories}")

#         binning_info[feature] = {
#                 'type': 'categorical',
#                 'top_categories': top_categories,
#                 'bin_ranges': {}  # No ranges for categorical
#             }

#     return binning_info


# def apply_binning(df: pd.DataFrame,
#                   feature: str,
#                   binning_info: Dict) -> pd.Series:
#     """
#     Apply binning to a feature based on binning information.

#     Parameters:
#     -----------
#     df : pd.DataFrame
#         Input dataframe
#     feature : str
#         Feature name
#     binning_info : Dict
#         Binning information for the feature

#     Returns:
#     --------
#     pd.Series with binned values
#     """
#     if binning_info['type'] == 'numerical':
#         if binning_info['bins'] is None:
#             return pd.Series(['Missing'] * len(df), index=df.index)

#         bins = binning_info['bins']
#         labels = [f"Bin_{i+1}" for i in range(len(bins)-1)]

#         binned = pd.cut(df[feature],
#                        bins=bins,
#                        labels=labels,
#                        include_lowest=True,
#                        duplicates='drop')

#         # Handle nulls - convert to string and then replace
#         binned = binned.astype(str)
#         binned[df[feature].isna()] = 'Missing'

#         return binned

#     else:  # categorical
#         top_cats = binning_info['top_categories']

#         # Convert to string for consistent comparison
#         if pd.api.types.is_categorical_dtype(df[feature]):
#             feature_data = df[feature].astype(str)
#         else:
#             feature_data = df[feature].astype(str)

#         # Replace NaN string representation with 'Missing'
#         feature_data = feature_data.replace('nan', 'Missing')

#         # Convert top_cats to strings for comparison
#         top_cats_str = [str(cat) for cat in top_cats]

#         # Apply binning logic: use category name if in top_cats, else 'Others' (except for Missing)
#         binned = feature_data.apply(lambda x: x if x in top_cats_str else ('Others' if x != 'Missing' else 'Missing'))

#         return binned


# def calculate_psi(expected_pct: pd.Series,
#                   actual_pct: pd.Series,
#                   epsilon: float = 0.0001) -> float:
#     """
#     Calculate Population Stability Index with proper epsilon handling and renormalization.

#     Parameters:
#     -----------
#     expected_pct : pd.Series
#         Expected (baseline) percentages
#     actual_pct : pd.Series
#         Actual percentages
#     epsilon : float
#         Small value to avoid log(0)

#     Returns:
#     --------
#     PSI value
#     """
#     # Align indices
#     all_bins = expected_pct.index.union(actual_pct.index)
#     expected_pct = expected_pct.reindex(all_bins, fill_value=0)
#     actual_pct = actual_pct.reindex(all_bins, fill_value=0)

#     # Only add epsilon where values are zero
#     expected_pct = expected_pct.apply(lambda x: epsilon if x == 0 else x)
#     actual_pct = actual_pct.apply(lambda x: epsilon if x == 0 else x)

#     # Renormalize to ensure they sum to 1 after adding epsilon
#     expected_pct = expected_pct / expected_pct.sum()
#     actual_pct = actual_pct / actual_pct.sum()

#     # Calculate PSI
#     psi_value = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))

#     return psi_value


# def calculate_month_on_month_psi(df: pd.DataFrame,
#                                  feature_list: List[str],
#                                  segment_columns: List[str],
#                                  month_col: str = 'Application_month',
#                                  data_selection_col: str = 'Data_selection',
#                                  account_id_col: str = 'digitalLoanAccountId') -> pd.DataFrame:
#     """
#     Calculate PSI for each feature comparing training period (June 2024 to March 2025)
#     vs each month after March 2025, overall and by segments.

#     Parameters:
#     -----------
#     df : pd.DataFrame
#         Input dataframe
#     feature_list : List[str]
#         List of features to calculate PSI for
#     segment_columns : List[str]
#         List of segment columns
#     month_col : str
#         Name of month column
#     data_selection_col : str
#         Name of data selection column (identifies train period)
#     account_id_col : str
#         Name of account ID column for counting distinct accounts

#     Returns:
#     --------
#     pd.DataFrame with PSI values with one row per feature-month-segment combination
#     """
#     # Create a copy to avoid modifying original
#     df = df.copy()

#     # Identify training and test periods
#     train_df = df[df[data_selection_col] == 'Train'].copy()
#     test_df = df[df[data_selection_col] != 'Train'].copy()

#     if len(train_df) == 0:
#         raise ValueError("No training data found. Check Data_selection column.")

#     print(f"Training period: {train_df[month_col].min()} to {train_df[month_col].max()}")
#     print(f"Test period: {test_df[month_col].min()} to {test_df[month_col].max()}")

#     # Identify feature types
#     feature_types = identify_feature_types(df, feature_list)

#     # Create binning strategy based on training period
#     binning_info = create_bins_for_features(
#         df,
#         feature_types['numerical'],
#         feature_types['categorical'],
#         train_df
#     )

#     # Get sorted test months
#     test_months = sorted(test_df[month_col].unique())

#     results = []

#     # Calculate overall PSI
#     for feature in feature_list:
#         if feature not in df.columns:
#             continue

#         # Apply binning to entire dataset
#         df[f'{feature}_binned'] = apply_binning(df, feature, binning_info[feature])
#         # print(f"Feature binned {df[f'{feature}_binned']}")
#         # Get training period distribution (baseline)
#         train_baseline = df[df[data_selection_col] == 'Train'][f'{feature}_binned'].value_counts(normalize=True)

#         # Calculate PSI for each test month
#         for month in test_months:
#             actual_dist = df[df[month_col] == month][f'{feature}_binned'].value_counts(normalize=True)
#             psi_value = calculate_psi(train_baseline, actual_dist)

#             # Calculate average percentages across all bins
#             expected_avg_pct = train_baseline.mean() * 100
#             actual_avg_pct = actual_dist.mean() * 100

#             # # Count distinct accounts for segment
#             # base_segment_count = train_segment[account_id_col].nunique()
#             # actual_segment_count = actual_segment[account_id_col].nunique()


#             results.append({
#                 'Feature': feature,
#                 'Feature_Type': binning_info[feature]['type'],
#                 'Segment_Column': 'Overall',
#                 'Segment_Value': 'All',
#                 'Month': f"{month}",
#                 'Base_Month': 'Train (Jun 2024 - Mar 2025)',
#                 'Current_Month': month,
#                 'Expected_Percentage': expected_avg_pct,
#                 'Actual_Percentage': actual_avg_pct,
#                 'PSI': psi_value
#             })

#     # Calculate PSI by segments
#     for segment_col in segment_columns:
#         if segment_col not in df.columns:
#             continue

#         segments = df[segment_col].dropna().unique()

#         for segment_val in segments:
#             segment_df = df[df[segment_col] == segment_val]

#             for feature in feature_list:
#                 if feature not in df.columns:
#                     continue

#                 # Get training period distribution for segment
#                 train_segment = segment_df[segment_df[data_selection_col] == 'Train']
#                 if len(train_segment) == 0:
#                     continue

#                 train_baseline = train_segment[f'{feature}_binned'].value_counts(normalize=True)

#                 # Calculate PSI for each test month
#                 for month in test_months:
#                     actual_segment = segment_df[segment_df[month_col] == month]
#                     if len(actual_segment) == 0:
#                         continue

#                     actual_dist = actual_segment[f'{feature}_binned'].value_counts(normalize=True)
#                     psi_value = calculate_psi(train_baseline, actual_dist)

#                     # Calculate average percentages across all bins
#                     expected_avg_pct = train_baseline.mean() * 100
#                     actual_avg_pct = actual_dist.mean() * 100

#                     # Count distinct accounts for segment
#                     base_segment_count = train_segment[account_id_col].nunique()
#                     actual_segment_count = actual_segment[account_id_col].nunique()

#                     results.append({
#                         'Feature': feature,
#                         'Feature_Type': binning_info[feature]['type'],
#                         'Segment_Column': segment_col,
#                         'Segment_Value': segment_val,
#                         'Month': f"{month}",
#                         'Base_Month': 'Train (Jun 2024 - Mar 2025)',
#                         'Current_Month': month,
#                         'Base_Count': base_segment_count,
#                         'Actual_Count': actual_segment_count,
#                         'Expected_Percentage': expected_avg_pct,
#                         'Actual_Percentage': actual_avg_pct,
#                         'PSI': psi_value
#                     })

#     return pd.DataFrame(results)


# def calculate_bin_level_psi(df: pd.DataFrame,
#                             feature_list: List[str],
#                             segment_columns: List[str],
#                             month_col: str = 'Application_month',
#                             data_selection_col: str = 'Data_selection',
#                             account_id_col: str = 'digitalLoanAccountId') -> pd.DataFrame:
#     """
#     Calculate bin-level PSI for each feature comparing training period
#     vs each month after March 2025, overall and by segments.

#     Parameters:
#     -----------
#     df : pd.DataFrame
#         Input dataframe
#     feature_list : List[str]
#         List of features to calculate PSI for
#     segment_columns : List[str]
#         List of segment columns
#     month_col : str
#         Name of month column
#     data_selection_col : str
#         Name of data selection column
#     account_id_col : str
#         Name of account ID column for counting distinct accounts

#     Returns:
#     --------
#     pd.DataFrame with bin-level PSI details including bin ranges
#     """
#     # Create a copy to avoid modifying original
#     df = df.copy()

#     # Identify training and test periods
#     train_df = df[df[data_selection_col] == 'Train'].copy()
#     test_df = df[df[data_selection_col] != 'Train'].copy()

#     if len(train_df) == 0:
#         raise ValueError("No training data found. Check Data_selection column.")

#     print(f"Training period: {train_df[month_col].min()} to {train_df[month_col].max()}")
#     print(f"Test period: {test_df[month_col].min()} to {test_df[month_col].max()}")

#     # Identify feature types
#     feature_types = identify_feature_types(df, feature_list)

#     # Create binning strategy based on training period
#     binning_info = create_bins_for_features(
#         df,
#         feature_types['numerical'],
#         feature_types['categorical'],
#         train_df
#     )

#     # Get sorted test months
#     test_months = sorted(test_df[month_col].unique())

#     results = []
#     epsilon = 0.0001

#     # Calculate overall bin-level PSI
#     for feature in feature_list:
#         if feature not in df.columns:
#             continue

#         # Apply binning to entire dataset
#         df[f'{feature}_binned'] = apply_binning(df, feature, binning_info[feature])
#         # print(df[f'{feature}_binned'])

#         # Get training period distribution (baseline)
#         train_baseline = df[df[data_selection_col] == 'Train'][f'{feature}_binned'].value_counts(normalize=True)

#         # Calculate bin-level PSI for each test month
#         for month in test_months:
#             month_data = df[df[month_col] == month]
#             actual_dist = month_data[f'{feature}_binned'].value_counts(normalize=True)

#             # Count distinct accounts
#             base_count = df[df[data_selection_col] == 'Train'][account_id_col].nunique()
#             actual_count = month_data[account_id_col].nunique()

#             # Get all bins
#             all_bins = train_baseline.index.union(actual_dist.index)

#             for bin_name in all_bins:
#                 # Simplified epsilon logic - no redundancy
#                 expected_pct = train_baseline.get(bin_name, 0)
#                 actual_pct = actual_dist.get(bin_name, 0)

#                 # Add epsilon only if zero
#                 expected_pct = epsilon if expected_pct == 0 else expected_pct
#                 actual_pct = epsilon if actual_pct == 0 else actual_pct

#                 # Calculate bin-level PSI
#                 bin_psi = (actual_pct - expected_pct) * np.log(actual_pct / expected_pct)

#                 # Get bin range information
#                 bin_ranges = binning_info[feature]['bin_ranges']
#                 if bin_name in bin_ranges:
#                     bin_min = bin_ranges[bin_name]['min']
#                     bin_max = bin_ranges[bin_name]['max']
#                     bin_range = bin_ranges[bin_name]['range_str']
#                 else:
#                     # For categorical or special bins (Missing, Others)
#                     bin_min = None
#                     bin_max = None
#                     bin_range = bin_name

#                 results.append({
#                     'Feature': feature,
#                     'Feature_Type': binning_info[feature]['type'],
#                     'Segment_Column': 'Overall',
#                     'Segment_Value': 'All',
#                     'Month': f"{month}",
#                     'Base_Month': 'Train (Jun 2024 - Mar 2025)',
#                     'Current_Month': month,
#                     'Base_Count': base_count,
#                     'Actual_Count': actual_count,
#                     'Bin': bin_name,
#                     'Bin_Range': bin_range,
#                     'Bin_Min': bin_min,
#                     'Bin_Max': bin_max,
#                     'Base_Percentage': (train_baseline.get(bin_name, 0) * 100),
#                     'Actual_Percentage': (actual_dist.get(bin_name, 0) * 100),
#                     'Bin_PSI': bin_psi
#                 })

#     # Calculate bin-level PSI by segments
#     for segment_col in segment_columns:
#         if segment_col not in df.columns:
#             continue

#         segments = df[segment_col].dropna().unique()

#         for segment_val in segments:
#             segment_df = df[df[segment_col] == segment_val]

#             for feature in feature_list:
#                 if feature not in df.columns:
#                     continue

#                 # Get training period distribution for segment
#                 train_segment = segment_df[segment_df[data_selection_col] == 'Train']
#                 if len(train_segment) == 0:
#                     continue

#                 train_baseline = train_segment[f'{feature}_binned'].value_counts(normalize=True)

#                 # Calculate bin-level PSI for each test month
#                 for month in test_months:
#                     actual_segment = segment_df[segment_df[month_col] == month]
#                     if len(actual_segment) == 0:
#                         continue

#                     actual_dist = actual_segment[f'{feature}_binned'].value_counts(normalize=True)

#                     # Count distinct accounts for segment
#                     base_segment_count = train_segment[account_id_col].nunique()
#                     actual_segment_count = actual_segment[account_id_col].nunique()

#                     # Get all bins
#                     all_bins = train_baseline.index.union(actual_dist.index)

#                     for bin_name in all_bins:
#                         # Simplified epsilon logic - no redundancy
#                         expected_pct = train_baseline.get(bin_name, 0)
#                         actual_pct = actual_dist.get(bin_name, 0)

#                         # Add epsilon only if zero
#                         expected_pct = epsilon if expected_pct == 0 else expected_pct
#                         actual_pct = epsilon if actual_pct == 0 else actual_pct

#                         # Calculate bin-level PSI
#                         bin_psi = (actual_pct - expected_pct) * np.log(actual_pct / expected_pct)

#                         # Get bin range information
#                         bin_ranges = binning_info[feature]['bin_ranges']
#                         if bin_name in bin_ranges:
#                             bin_min = bin_ranges[bin_name]['min']
#                             bin_max = bin_ranges[bin_name]['max']
#                             bin_range = bin_ranges[bin_name]['range_str']
#                         else:
#                             # For categorical or special bins (Missing, Others)
#                             bin_min = None
#                             bin_max = None
#                             bin_range = bin_name

#                         results.append({
#                             'Feature': feature,
#                             'Feature_Type': binning_info[feature]['type'],
#                             'Segment_Column': segment_col,
#                             'Segment_Value': segment_val,
#                             'Month': f"{month}",
#                             'Base_Month': 'Train (Jun 2024 - Mar 2025)',
#                             'Current_Month': month,
#                             'Base_Count': base_segment_count,
#                             'Actual_Count': actual_segment_count,
#                             'Bin': bin_name,
#                             'Bin_Range': bin_range,
#                             'Bin_Min': bin_min,
#                             'Bin_Max': bin_max,
#                             'Base_Percentage': (train_baseline.get(bin_name, 0) * 100),
#                             'Actual_Percentage': (actual_dist.get(bin_name, 0) * 100),
#                             'Bin_PSI': bin_psi
#                         })

#     return pd.DataFrame(results)

# End